In [ ]:
# Databricks notebook source
# 06_clean_vigigrip

import hashlib
import re
import unicodedata
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from pyspark.sql.types import DateType, DoubleType, StringType, StructField, StructType

CATALOG = "decide_catalog"
SCHEMA = "decide_schema"
VOLUME_PATH = "/Volumes/decide_catalog/decide_schema/decide_volume"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")


def should_run_pipeline():
    try:
        value = dbutils.jobs.taskValues.get(taskKey="00_check_source_changes", key="should_run", default="true")
        return str(value).lower() == "true"
    except Exception:
        return True


def source_path(file_name):
    return f"{VOLUME_PATH}/{file_name}"


def sha256_hash(value):
    if pd.isna(value):
        return None
    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()


def month_start(values):
    return pd.to_datetime(values, errors="coerce").dt.to_period("M").dt.start_time.dt.date


def week_start(values, week_start="sunday"):
    dates = pd.to_datetime(values, errors="coerce")
    freq = "W-SAT" if week_start == "sunday" else "W-SUN"
    return dates.dt.to_period(freq).dt.start_time.dt.date


def max_with_na(series):
    values = pd.to_numeric(series, errors="coerce")
    if values.notna().any():
        return values.max(skipna=True)
    return np.nan


def write_delta(pdf, table_name):
    output_cols = [
        "Lab_reference",
        "Country",
        "Breed",
        "Province",
        "Farm_ID",
        "Diagnostic_test",
        "Sample_type",
        "Samplenumber",
        "Date",
        "Date_month",
        "Date_week",
        "Pathogen",
        "Result",
    ]
    pdf = pdf.reindex(columns=output_cols).copy()

    string_cols = [
        "Lab_reference",
        "Country",
        "Breed",
        "Province",
        "Farm_ID",
        "Diagnostic_test",
        "Sample_type",
        "Samplenumber",
        "Pathogen",
    ]
    for col in string_cols:
        pdf[col] = pdf[col].where(pd.notna(pdf[col]), None).astype(object)
    for col in ["Date", "Date_month", "Date_week"]:
        pdf[col] = pd.to_datetime(pdf[col], errors="coerce").dt.date
    pdf["Result"] = pd.to_numeric(pdf["Result"], errors="coerce")

    schema = StructType(
        [
            StructField("Lab_reference", StringType(), True),
            StructField("Country", StringType(), True),
            StructField("Breed", StringType(), True),
            StructField("Province", StringType(), True),
            StructField("Farm_ID", StringType(), True),
            StructField("Diagnostic_test", StringType(), True),
            StructField("Sample_type", StringType(), True),
            StructField("Samplenumber", StringType(), True),
            StructField("Date", DateType(), True),
            StructField("Date_month", DateType(), True),
            StructField("Date_week", DateType(), True),
            StructField("Pathogen", StringType(), True),
            StructField("Result", DoubleType(), True),
        ]
    )
    sdf = spark.createDataFrame(pdf, schema=schema)
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    (
        sdf.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_name)
    )
    print(f"Wrote {sdf.count()} rows to {full_name}")

def region_lookup_key(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace("?", "")
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text or None


REGION_TO_PROVINCE = {
    "bretagne": "Brittany",
    "nouvelle aquitaine": "Nouvelle Aquitaine",
    "auvergne rhone alpes": "Auvergne-Rhône-Alpes",
    "bourgogne franche comte": "Bourgogne-Franche-Comté",
    "centre val de loire": "Centre-Val de Loire",
    "drom": "DROM",
    "grand est": "Grand Est",
    "hauts de france": "Hauts-de-France",
    "ile de france": "Ile de France",
    "normandie": "Normandy",
    "occitanie": "Occitanie",
    "pays de loire": "Pays de la Loire",
    "provence alpes cotes d azur": "Provence-Alpes-Côte d'Azur",
    "provence alpes cote d azur": "Provence-Alpes-Côte d'Azur",
}


def map_french_province(region):
    key = region_lookup_key(region)
    if key is None:
        return "Missing"
    return REGION_TO_PROVINCE.get(key, "Missing")


if not should_run_pipeline():
    dbutils.notebook.exit("No source changes detected; skipping Vigigrip")

raw = pd.read_excel(source_path("DECIDE_final_version.xlsx"), engine="openpyxl")
df = raw.rename(columns={"Dossier.ID": "Filenumber", "Sample.ID": "Samplenumber", "Herd.ID": "Farm_ID", "Reason.of.sampling": "Project"})
df["Country"] = "France"
df["Lab_reference"] = "6"
df["Sample_type"] = df["Sample_type"].map({"Lung": "Autopsy", "BAL": "BAL", "Swab": "Swab", "TTA": "TTA"}).fillna("Missing")
df["Breed"] = df["Breed"].map({"Beef": "Beef", "Dairy": "Dairy", "Mixed": "Mixed", "Veal Calf": "Veal"}).fillna("Unknown")
df["Province"] = df["Region"].map(map_french_province)
unmapped = df.loc[df["Province"].eq("Missing"), "Region"].value_counts()
if len(unmapped):
    print("Unmapped Region values (top 20):")
    print(unmapped.head(20).to_string())
df["Diagnostic_test"] = df["Diagnostic_test"].replace({"CULTURE": "Culture"})
pathogen_cols = ["PM", "MH", "HS", "MB", "BRSV", "PI3", "BCV"]
df = df[["Filenumber", "Diagnostic_test", "Samplenumber", "Country", "Lab_reference", "Sample_type", "Breed", *pathogen_cols, "Date", "Province", "Project", "Farm_ID"]].drop_duplicates()
for col in ["Filenumber", "Samplenumber", "Farm_ID"]:
    df[col] = df[col].apply(sha256_hash)
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Date_month"] = month_start(df["Date"])
df["Date_week"] = week_start(df["Date"], week_start="sunday")
df = df[(df["Province"] != "DROM") & (df["Project"] == "Respiratory disease")]
group_cols = ["Lab_reference", "Country", "Breed", "Province", "Farm_ID", "Diagnostic_test", "Sample_type", "Samplenumber", "Date_month", "Date_week", "Date"]
grouped = df.groupby(group_cols, dropna=False)[pathogen_cols].agg(max_with_na).reset_index()
barometer = grouped.melt(id_vars=group_cols, value_vars=pathogen_cols, var_name="Pathogen", value_name="Result")
barometer = barometer[["Lab_reference", "Country", "Breed", "Province", "Farm_ID", "Diagnostic_test", "Sample_type", "Samplenumber", "Date", "Date_month", "Date_week", "Pathogen", "Result"]]
write_delta(barometer, "barometer_vigigrip")
